## 7b. Our results vs the papers
Two comparisons, each in atomic cells (re-run after more runs have finished):
* **7b.1 FedUP: ours vs the FedUP paper**: the main table (R-A, F-A, 0A, PS, MIA), time and communication, the four ablations, and an automatic scoreboard of the paper's claims.
* **7b.2 FuGuard vs FedUP, on the FuGuard paper's protocol**: FuGuard's numbers from the FuGuard paper (plus our own FuGuard reproduction) next to FedUP run by us under that protocol.

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

### 7b.1 FedUP: ours vs the FedUP paper

In [ ]:
from fedup_repro import comparison as C, plotting as PL
from IPython.display import Image, display
out = C.fedup_vs_paper()
display(out['summary'].round(3))
if len(out['main']):
    display(Image(str(PL.plot_fedup_parity(out['main']))))

In [ ]:
from fedup_repro import comparison as C
m = C.main_vs_paper()
display(m.sort_values('abs_delta', ascending=False).head(30))

In [ ]:
from fedup_repro import comparison as C
display(C.cost_vs_paper())

In [ ]:
from fedup_repro import comparison as C
for name in ['dim', 'alpha', 'dp', 'centroid']:
    print(name)
    display(getattr(C, name + '_vs_paper')())

In [ ]:
from fedup_repro import comparison as C
display(C.claims_scoreboard())

### 7b.2 FuGuard vs FedUP (FuGuard paper's protocol)
How to read it:
* **ASR** is shown in **FuGuard's definition** (share of the triggered test set predicted as class 9, which includes the untouched class-9 images and therefore has a floor); the **strict** ASR (non-target inputs only) is listed next to it.
* **FedUP has no recovery rounds**, so FuGuard's recovery-round metric has no FedUP counterpart; FedUP is measured right after its one-shot unlearning.
* **MIA** is not compared: the two papers define it differently and FuGuard reports it only in figures.
* **FedUP costs differ in kind**: every retained client uploads class centroids once, while FuGuard needs the target client plus recovery rounds.
* EuroSAT: the train/test split is a fixed 80/20 here (FuGuard splits with the run seed), and FuGuard's released trigger for EuroSAT is gray, which is reproduced on purpose (`bd_trigger="repo"`).
* "FuGuard (our repro)" is the FuGuard reproduction from Weekly Update 2, listed as a sanity reference.

In [ ]:
from fedup_repro import comparison as C
display(C.fuguard_vs_fedup_compact())

In [ ]:
from fedup_repro import comparison as C, plotting as PL
from IPython.display import Image, display
fv = C.fuguard_vs_fedup()
display(fv)
if 'fedup_ACC' in fv and fv['fedup_ACC'].notna().any():
    display(Image(str(PL.plot_fuguard_vs_fedup(fv.dropna(subset=['fedup_ACC'])))))